## Train on CelebA

In [ ]:
from train import Trainer
from config import config
from model import UNet
from modules import NoiseInjectionPipeline
from denoising_diffusion import NoisePredictionPipeline, DDPM

# Forward diffusion
noise_injection_pipeline = NoiseInjectionPipeline(
    config=config,
    random=True
)

# Noise prediction backbone
unet = UNet(
    in_channels = config["in_channels"], # because not it will operate in latent space and not in RGB Space
    out_channels = config["out_channels"],
    channel_multipliers = (1, 2, 4, 8),
    time_embd_dim = config["time_embed"],
    base_channels = 64
)

# Noise prediction pipeline
noise_prediction_pipeline = NoisePredictionPipeline(
    config=config,
    noise_prediction_model=unet
)

# Complete DDPM
ddpm = DDPM(
    noise_injection_pipieline=noise_injection_pipeline,
    noise_prediction_pipeline=noise_prediction_pipeline
)

trainer = Trainer(
    ddpm=ddpm,
    config=config,

    dataset_path="../data/celeba/celeba_latents.pt",

    batch_size = config["batch_size"],
    image_size = config["image_size"],

    epochs = 350,
    warmup_epochs = 10,
    learning_rate = 1e-4,
    use_min_snr_loss = config["use_min_snr_loss"],

    validation_split = 0.05,

    checkpoint_dir="./checkpoint/celeba_256",
    checkpoint_every = 10,

    project_name="DiffusionEngine",
    run_name="DDPM-CelebA-HQ-256x256",

    num_workers = 4,
    use_amp = False
)
trainer.resume(checkpoint_path = "./checkpoint/celeba_256/latest.pt")
trainer.fit()

Loaded latent dataset: 23429 images
Latent shape: (23429, 4, 32, 32)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Himanshu Singh\_netrc.


Fixed validation noise and timesteps generated.


wandb: Currently logged in as: himanshusr451tehs (himanshusr451tehs-personaluse-com) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


DDPM TRAINER INITIALIZED
Device:          cuda
Mixed precision: False
Latent Denoise : Enabled
Precomputed Latents: Enabled
Loss: MIN-SNR-Loss
Dataset size:    23429
Training images: 22258
Validation:      1171
Batch size:      16
Image size:      256x256
Training steps:  1391 per epoch
W&B run:         https://wandb.ai/himanshusr451tehs-personaluse-com/DiffusionEngine/runs/o7v9ylhf
TRAINING RESUMED
Starting epoch: 193
Global step:    267072
Best val loss:  0.080017


Epoch 193/350:  89%|████████▊ | 1233/1391 [02:52<01:13,  2.15it/s, loss=0.069026, lr=5.00e-05]

##

In [ ]:
import torch
from sample import sample
import matplotlib.pyplot as plt
from diffusers import AutoencoderKL

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = AutoencoderKL.from_pretrained(
        config["vae_model"]
    ).to(device)

vae.eval()
vae.requires_grad_(False)

generated_image = sample(ddpm, config, device, batch_size=config["batch_size"], vae = vae)
generated_image = (generated_image + 1.0) / 2.0 # Convert [-1, 1] -> [0, 1]
generated_image = generated_image.clamp(0, 1) # for visualization in matplotlib

# Visualize

# Convert BCHW -> HWC
image = generated_image[0].permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Generated Image")
plt.show()

C:\Users\Himanshu Singh\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 7.60 GiB is allocated by PyTorch, and 1.14 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)